In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer  # ПРАВИЛЬНЫЙ ИМПОРТ!
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)
from sklearn.decomposition import PCA
import json
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DATA_DIR = Path('data')
ARTIFACTS_DIR = Path('artifacts')
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
LABELS_DIR = ARTIFACTS_DIR / 'labels'

ARTIFACTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
LABELS_DIR.mkdir(exist_ok=True)

all_results = {}
best_configs = {}

# Список датасетов
datasets = ['S07-hw-dataset-01', 'S07-hw-dataset-02', 'S07-hw-dataset-03']

for ds_name in datasets:
    print(f"=== Обработка {ds_name} ===")
    
    # Загрузка данных
    df = pd.read_csv(DATA_DIR / f'{ds_name}.csv')
    sample_ids = df['sample_id']
    X_raw = df.drop(columns=['sample_id'])
    
    # Препроцессинг: только числовые признаки
    numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) < X_raw.shape[1]:
        print(f"⚠️  В {ds_name} есть нечисловые признаки — используем только числовые.")
        X_raw = X_raw[numeric_cols]
    
    # Проверка пропусков
    if X_raw.isnull().sum().sum() > 0:
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X_raw)
    else:
        X_imputed = X_raw.values
    
    # Масштабирование
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)
    
    # === KMeans ===
    k_range = range(2, 11)
    sil_scores = []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        sil_scores.append(silhouette_score(X_scaled, labels))
    
    best_k = k_range[np.argmax(sil_scores)]
    kmeans_best = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
    labels_kmeans = kmeans_best.fit_predict(X_scaled)
    
    # === DBSCAN ===
    min_samples = max(5, int(0.01 * len(X_scaled)))
    eps_candidates = [0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
    best_sil_db = -1
    best_eps = None
    best_labels_db = None
    
    for eps in eps_candidates:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels_db = dbscan.fit_predict(X_scaled)
        n_noise = np.sum(labels_db == -1)
        noise_ratio = n_noise / len(labels_db)
        unique_labels = set(labels_db)
        if noise_ratio > 0.6 or len(unique_labels) < 2 or (-1 in unique_labels and len(unique_labels) == 2):
            continue
        try:
            sil = silhouette_score(X_scaled[labels_db != -1], labels_db[labels_db != -1])
            if sil > best_sil_db:
                best_sil_db = sil
                best_eps = eps
                best_labels_db = labels_db.copy()
        except:
            continue
    
    # === Метрики ===
    res = {}
    res['KMeans'] = {
        'k': int(best_k),
        'silhouette': float(silhouette_score(X_scaled, labels_kmeans)),
        'davies_bouldin': float(davies_bouldin_score(X_scaled, labels_kmeans)),
        'calinski_harabasz': float(calinski_harabasz_score(X_scaled, labels_kmeans)),
        'noise_ratio': 0.0
    }
    
    if best_labels_db is not None:
        non_noise = best_labels_db != -1
        res['DBSCAN'] = {
            'eps': float(best_eps),
            'min_samples': int(min_samples),
            'silhouette': float(silhouette_score(X_scaled[non_noise], best_labels_db[non_noise])),
            'davies_bouldin': float(davies_bouldin_score(X_scaled[non_noise], best_labels_db[non_noise])),
            'calinski_harabasz': float(calinski_harabasz_score(X_scaled[non_noise], best_labels_db[non_noise])),
            'noise_ratio': float(np.mean(best_labels_db == -1))
        }
    else:
        res['DBSCAN'] = None
    
    # === Выбор лучшего метода ===
    best_method = 'KMeans'
    best_labels = labels_kmeans
    if res['DBSCAN'] and res['DBSCAN']['silhouette'] > res['KMeans']['silhouette']:
        best_method = 'DBSCAN'
        best_labels = best_labels_db
    
    # === Визуализация ===
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)
    
    plt.figure(figsize=(6, 5))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels, cmap='tab10', alpha=0.7)
    plt.title(f'{ds_name}: PCA + {best_method}')
    plt.savefig(FIGURES_DIR / f'pca_{ds_name}.png')
    plt.close()
    
    plt.figure(figsize=(6, 4))
    plt.plot(k_range, sil_scores, marker='o')
    plt.axvline(best_k, color='r', linestyle='--', label=f'Best k={best_k}')
    plt.title(f'{ds_name}: Silhouette vs k')
    plt.legend()
    plt.savefig(FIGURES_DIR / f'silhouette_vs_k_{ds_name}.png')
    plt.close()
    
    # === Сохранение ===
    pd.DataFrame({'sample_id': sample_ids, 'cluster_label': best_labels}).to_csv(
        LABELS_DIR / f'labels_{ds_name}.csv', index=False
    )
    
    all_results[ds_name] = res
    best_configs[ds_name] = {'best_method': best_method, 'params': res[best_method]}

=== Обработка S07-hw-dataset-01 ===
=== Обработка S07-hw-dataset-02 ===
=== Обработка S07-hw-dataset-03 ===


In [7]:
print("\n=== Устойчивость KMeans на Dataset-01 ===")
df1 = pd.read_csv(DATA_DIR / 'S07-hw-dataset-01.csv')
X1 = df1.drop(columns=['sample_id'])
scaler1 = StandardScaler()
X1_scaled = scaler1.fit_transform(X1)

# Определяем k из результатов
k1 = all_results['S07-hw-dataset-01']['KMeans']['k']
ari_list = []
ref_labels = KMeans(n_clusters=k1, random_state=42, n_init=10).fit_predict(X1_scaled)
for seed in range(5):
    new_labels = KMeans(n_clusters=k1, random_state=seed, n_init=10).fit_predict(X1_scaled)
    ari_list.append(adjusted_rand_score(ref_labels, new_labels))

print(f"ARI между запусками: {[round(x, 3) for x in ari_list]}")
print(f"Среднее ARI: {np.mean(ari_list):.3f} ± {np.std(ari_list):.3f}")

# Ячейка 3: Сохранение артефактов
with open(ARTIFACTS_DIR / 'metrics_summary.json', 'w') as f:
    json.dump(all_results, f, indent=4)
with open(ARTIFACTS_DIR / 'best_configs.json', 'w') as f:
    json.dump(best_configs, f, indent=4)

print("\n HW07 успешно завершён для всех 3 датасетов!")


=== Устойчивость KMeans на Dataset-01 ===
ARI между запусками: [1.0, 1.0, 1.0, 1.0, 1.0]
Среднее ARI: 1.000 ± 0.000

 HW07 успешно завершён для всех 3 датасетов!
